# 02. Pandas 기초
**목표**: 표 데이터를 읽고, 살펴보고, 고치고, 집계하는 방법을 익힙니다. (예상 10~12시간)

이 미션에서 가장 많이 쓰는 도구입니다. 아래 7가지를 순서대로 다룹니다.

1. 데이터 만들기/살펴보기 2. 선택·필터링 3. 결측치 4. GroupBy 5. 날짜 6. 문자열 7. 새 컬럼과 merge

연습용 가짜 상품 데이터를 직접 만들어서 사용합니다. (실제 CSV는 `pd.read_csv("경로")` 로 읽습니다.)

In [3]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n = 200
df = pd.DataFrame({
    "product_id": np.arange(1, n + 1),
    "name": rng.choice(
        ["Red Cotton T-Shirt", "Blue Denim Jeans", "Classic White Sneakers",
         "Black Leather Wallet", "Slim Fit Chino Pants", "Wool Winter Coat", "Hat"], n),
    "category": rng.choice(["상의", "하의", "신발", "잡화", "아우터"], n),
    "price": rng.normal(50000, 15000, n).round(-2),
    "rating": rng.choice([3.0, 3.5, 4.0, 4.5, 5.0], n),
    "created_at": pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n), unit="D"),
})
# 일부러 결측치와 이상치 만들기
df.loc[rng.choice(n, 15, replace=False), "price"] = np.nan
df.loc[rng.choice(n, 10, replace=False), "rating"] = np.nan
df.loc[3, "price"] = 900000     # 이상치
print(df.shape)

(200, 6)


## 1. 처음 살펴보기: `head`, `info`, `describe` (미션 4.1)

In [4]:
df.head()

,product_id,name,category,price,rating,created_at
0,1,Wool Winter Coat,하의,41200.0,4.0,2025-05-23
1,2,Slim Fit Chino Pants,신발,NaN,5.0,2025-06-26
2,3,Black Leather Wallet,하의,58800.0,5.0,2025-03-28
3,4,Blue Denim Jeans,하의,900000.0,3.5,2025-03-08
4,5,Classic White Sneakers,아우터,40800.0,3.5,2025-12-05


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   product_id  200 non-null    int64         
 1   name        200 non-null    str           
 2   category    200 non-null    str           
 3   price       185 non-null    float64       
 4   rating      190 non-null    float64       
 5   created_at  200 non-null    datetime64[us]
dtypes: datetime64[us](1), float64(2), int64(1), str(2)
memory usage: 9.5 KB


`info()` 읽는 법
- `Non-Null Count` 가 전체 행 수(200)보다 작으면 **결측치가 있다**는 뜻입니다.
- `Dtype` 이 의도와 다른지(날짜가 문자열 등) 확인합니다.

In [8]:
df.describe()

,product_id,price,rating,created_at
count,200.000000,185.000000,190.000000,200
mean,100.500000,52858.378378,3.997368,2025-07-02 02:09:36
min,1.000000,-6600.000000,3.000000,2025-01-03 00:00:00
25%,50.750000,37900.000000,3.500000,2025-04-07 00:00:00
50%,100.500000,46900.000000,4.000000,2025-06-27 00:00:00
75%,150.250000,58800.000000,4.500000,2025-09-21 06:00:00
max,200.000000,900000.000000,5.000000,2025-12-31 00:00:00
std,57.879185,64442.341498,0.673570,NaN


`describe()` 읽는 법
- `mean` 과 `50%`(중앙값) 차이가 크면 한쪽으로 치우친 분포나 이상치를 의심합니다.
- `max` 가 `75%` 에 비해 터무니없이 크면 이상치 후보입니다.

> 해석 예시: price의 평균은 약 5만 원인데 최대값이 90만 원으로 매우 커서 이상치가 있다. → 분석 전에 이상치 처리가 필요하다. (수치 근거 + 시사점)

## 2. 선택과 필터링

In [13]:
print(df["price"].head(3))            # 컬럼 하나 → Series
print(df[["name", "price"]].head(3))  # 컬럼 여러 개 → DataFrame

0    41200.0
1        NaN
2    58800.0
Name: price, dtype: float64
                   name    price
0      Wool Winter Coat  41200.0
1  Slim Fit Chino Pants      NaN
2  Black Leather Wallet  58800.0


In [15]:
expensive = df[df["price"] > 70000]
print(len(expensive), "개")

# 조건 여러 개: & (그리고), | (또는), 각 조건은 괄호로 감싸기
cond = (df["category"] == "신발") & (df["rating"] >= 4.5)
df[cond].head()

15 개


,product_id,name,category,price,rating,created_at
1,2,Slim Fit Chino Pants,신발,NaN,5.0,2025-06-26
10,11,Slim Fit Chino Pants,신발,30600.0,4.5,2025-01-16
11,12,Hat,신발,42900.0,5.0,2025-08-09
13,14,Slim Fit Chino Pants,신발,52000.0,4.5,2025-07-08
43,44,Red Cotton T-Shirt,신발,69500.0,4.5,2025-08-13


## 3. 결측치 다루기

In [17]:
print(df.isna().sum())                       # 컬럼별 결측 개수
print((df.isna().mean() * 100).round(1))     # 결측 비율(%)

product_id     0
name           0
category       0
price         15
rating        10
created_at     0
dtype: int64
product_id    0.0
name          0.0
category      0.0
price         7.5
rating        5.0
created_at    0.0
dtype: float64


### 대치 방법 비교
- 전체 평균/중앙값으로 채우기 (간단하지만 그룹 특성을 무시)
- **그룹별 중앙값으로 채우기** ← 미션 4.4 요구사항

In [21]:
df_a = df.copy()
df_a["price"] = df_a["price"].fillna(df_a["price"].median())   # 전체 중앙값
print("전체 중앙값 대치 후 결측:", df_a["price"].isna().sum())

전체 중앙값 대치 후 결측: 0


In [23]:
df_b = df.copy()
# groupby + transform: 각 행에 '자기 그룹의 중앙값'을 붙여 줌
group_median = df_b.groupby("category")["price"].transform("median")
df_b["price"] = df_b["price"].fillna(group_median)
print("그룹별 중앙값 대치 후 결측:", df_b["price"].isna().sum())

그룹별 중앙값 대치 후 결측: 0


`transform` 이 핵심입니다. `groupby().median()` 은 그룹 수만큼(5개) 결과를 주지만, `transform("median")` 은 **원래 행 수(200개)와 같은 길이**로 돌려줘서 `fillna` 에 바로 쓸 수 있습니다.

## 4. GroupBy

In [25]:
print(df_b.groupby("category")["price"].mean().round(0))

df_b.groupby("category").agg(
    상품수=("product_id", "count"),
    평균가격=("price", "mean"),
    최고가격=("price", "max"),
).round(0)

category
상의     46990.0
신발     48857.0
아우터    49156.0
잡화     49061.0
하의     72315.0
Name: price, dtype: float64


,상품수,평균가격,최고가격
category,,,
상의,39,46990.0,84700.0
신발,45,48857.0,96000.0
아우터,47,49156.0,85200.0
잡화,36,49061.0,70700.0
하의,33,72315.0,900000.0


## 5. 날짜 다루기

In [30]:
df_b["created_at"] = pd.to_datetime(df_b["created_at"])   # 문자열이면 변환 필요
df_b["month"] = df_b["created_at"].dt.month
df_b["weekday"] = df_b["created_at"].dt.day_name()
df_b[["created_at", "month", "weekday"]].head()

,created_at,month,weekday
0,2025-05-23,5,Friday
1,2025-06-26,6,Thursday
2,2025-03-28,3,Friday
3,2025-03-08,3,Saturday
4,2025-12-05,12,Friday


In [31]:
# 월별 집계 (라인차트의 재료)
monthly = df_b.set_index("created_at")["price"].resample("MS").sum()
monthly.head()

created_at
2025-01-01     667400.0
2025-02-01     557200.0
2025-03-01    1788100.0
2025-04-01    1172750.0
2025-05-01     545000.0
Freq: MS, Name: price, dtype: float64

`resample("MS")` 는 월 단위로 묶습니다 (MS = Month Start). 미션의 "월별 매출 추이" 라인차트에 그대로 쓰입니다.

## 6. 문자열 다루기 (미션 4.3 텍스트 피처)

In [14]:
df_b["name_len"] = df_b["name"].str.len()                  # 문자 수
df_b["word_count"] = df_b["name"].str.split().str.len()    # 단어 수
df_b[["name", "name_len", "word_count"]].drop_duplicates().head(7)

,name,name_len,word_count
0,Wool Winter Coat,16,3
1,Slim Fit Chino Pants,20,4
2,Black Leather Wallet,20,3
3,Blue Denim Jeans,16,3
4,Classic White Sneakers,22,3
5,Red Cotton T-Shirt,18,3
11,Hat,3,1


## 7. 새 컬럼 만들기와 merge

In [15]:
df_b["price_level"] = np.where(df_b["price"] >= 60000, "고가", "일반")
df_b["price_level"].value_counts()

price_level
일반    158
고가     42
Name: count, dtype: int64

In [16]:
# merge: 공통 키로 두 표를 합치기 (RFM에서 주문표와 고객표를 합칠 때 사용)
orders = pd.DataFrame({"order_id": [1, 2, 3], "product_id": [1, 2, 2], "qty": [1, 2, 1]})
orders.merge(df_b[["product_id", "name", "price"]], on="product_id", how="left")

,order_id,product_id,qty,name,price
0,1,1,1,Wool Winter Coat,41200.0
1,2,2,2,Slim Fit Chino Pants,46450.0
2,3,2,1,Slim Fit Chino Pants,46450.0


## 정리
| 하고 싶은 일 | 코드 |
|---|---|
| 구조 확인 | `info()`, `describe()`, `head()` |
| 조건 필터 | `df[(조건1) & (조건2)]` |
| 결측 확인 | `df.isna().sum()` |
| 그룹별 대치 | `fillna(groupby(...).transform("median"))` |
| 그룹 요약 | `groupby().agg(...)` |
| 월별 집계 | `set_index(날짜).resample("MS")` |
| 단어 수 | `str.split().str.len()` |

### ✏️ 직접 해보기
`rating` 결측치를 **카테고리별 평균**으로 채우고, 채운 뒤 결측이 0개인지 확인하세요.

In [ ]:
# 여기에 직접 작성해 보세요

**정답 예시** (먼저 직접 풀어본 뒤 확인하세요)

In [ ]:
d = df.copy()
d["rating"] = d["rating"].fillna(d.groupby("category")["rating"].transform("mean"))
print(d["rating"].isna().sum())

### ✏️ 직접 해보기
카테고리별로 상품 수와 평균 평점을 구하고, 평균 평점이 높은 순서로 정렬하세요.

> 힌트: `agg`, `sort_values(ascending=False)`

In [ ]:
# 여기에 직접 작성해 보세요

**정답 예시** (먼저 직접 풀어본 뒤 확인하세요)

In [ ]:
(df.groupby("category")
   .agg(상품수=("product_id", "count"), 평균평점=("rating", "mean"))
   .sort_values("평균평점", ascending=False))

✅ 다음: `03_statistics_basics`